# ML-04 - Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Divyanshu10045/flyrank-assignments/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

First time on the real warehouse release rather than the 30k starter CSV, and the one thing that
changes everything is that I can finally put my label in the future. Lane 2 is unchanged: rank the
pages a reviewer should look at first. What ML-03 could not do, this can - it measured a label
carved out of the same 90 days as its features, and said so. Here the features stop on 2026-02-28
and the label starts on 2026-03-01.

In [2]:
# Setup: dependencies, credentials, and the paths this notebook reads.
#
# This is a LOCAL run, not Colab. The release README suggests DuckDB; on this
# machine duckdb's binary is refused by a Windows Application Control policy
# ("DLL load failed ... has blocked this file"), so I read the Parquet with
# pyarrow instead and push every question down to the columns before anything is
# materialised. Same discipline the DuckDB examples teach, different engine.

import importlib
import os
import time
from pathlib import Path

REQUIRED = {"pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub",
            "python-dotenv": "dotenv", "pandas": "pandas", "numpy": "numpy",
            "scikit-learn": "sklearn"}
_missing = []
for _pip, _mod in REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ImportError:
        _missing.append(_pip)
if _missing:
    raise SystemExit("missing packages: %s -> pip install %s" % (_missing, " ".join(_missing)))

import dotenv
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem

# Climb to the repo root so the notebook runs the same from work/notebooks/ or the top.
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "work" / "notebooks").is_dir() and (p / ".git").exists()), None)
if REPO is None:
    raise SystemExit("could not find the repo root from %s" % Path.cwd())
os.chdir(REPO)
print("repo root:", REPO)
print("pandas %s | numpy %s | pyarrow %s" % (pd.__version__, np.__version__, pa.__version__))

# The read token lives in .env at the repo root, which .gitignore covers. It is
# never printed, never written into a cell, and never echoed back out.
dotenv.load_dotenv(REPO / ".env")
_HF = os.environ.get("HF_TOKEN", "")
if not _HF:
    raise SystemExit("HF_TOKEN not found. Put it in %s as HF_TOKEN=... and rerun." % (REPO / ".env"))
fs = HfFileSystem(token=_HF)
print("HF_TOKEN loaded: %s (length %d, value never printed)" % (bool(_HF), len(_HF)))

WAREHOUSE = "datasets/FlyRank/internship-warehouse"
FACT = WAREHOUSE + "/fact_content_daily_performance"


def month_file(m):
    """A month partition is a DIRECTORY of parquet parts, not a single file."""
    parts = sorted(p for p in fs.ls(FACT + "/month=" + m, detail=False) if p.endswith(".parquet"))
    if not parts:
        raise FileNotFoundError("no parquet under month=%s" % m)
    return parts[0]


def schema_of(path):
    """Column names straight from parquet metadata - costs nothing, reads no rows."""
    return set(pq.ParquetFile(fs.open(path)).metadata.schema.names)


# ---- my windows -----------------------------------------------------------
# 59 days of features, then a decision point, then 31 days of label. No overlap.
FEATURE_START, FEATURE_END = "2026-01-01", "2026-02-28"
LABEL_START, LABEL_END = "2026-03-01", "2026-03-31"
DECISION_POINT = LABEL_START
FEATURE_MONTHS = ("2026-01", "2026-02")     # the 59 days
LABEL_MONTHS = ("2026-03",)                 # the 31 days
DEV_MONTH = "2026-03"                       # the mid-panel month I develop on

PRIOR_DAYS = (pd.Timestamp(FEATURE_END) - pd.Timestamp(FEATURE_START)).days + 1
LABEL_DAYS = (pd.Timestamp(LABEL_END) - pd.Timestamp(LABEL_START)).days + 1
print("feature window %s -> %s (%d days) | decision point %s | label window %s -> %s (%d days)"
      % (FEATURE_START, FEATURE_END, PRIOR_DAYS, DECISION_POINT, LABEL_START, LABEL_END, LABEL_DAYS))
print("windows share days:", bool(set(pd.date_range(FEATURE_START, FEATURE_END))
                                 & set(pd.date_range(LABEL_START, LABEL_END))))

DAILY_COLS = ["report_date", "client_hash_id", "content_hash_id",
              "gsc_impressions", "gsc_clicks", "gsc_sum_position",
              "gsc_avg_position", "gsc_data_available"]
KEYS = ["client_hash_id", "content_hash_id"]
_AGG = {}
# Filled in by the first analysis cell, before anything aggregates. Until then the
# client filter is off, so a mis-ordered run fails loudly rather than silently
# returning an empty frame.
MY_CLIENTS = set()


def read_month(m, cols=None):
    return pq.read_table(fs.open(month_file(m)), columns=cols or DAILY_COLS)


def agg_month(m, table=None):
    """Per (client, content) sums over one month.

    Three filters, all deliberate: my own client set (the panel is unbalanced, so
    other clients' rows are not mine to model), gsc_data_available IS TRUE (the flag is
    three-valued, so IS TRUE is the only honest test), and gsc_impressions > 0 (a day with
    no impressions contributes nothing, and leaving it in would make 'active days' mean
    'rows present' rather than 'days the page actually showed up').
    Memoised so each partition is read at most once per run.
    """
    if m in _AGG:
        return _AGG[m]
    t = read_month(m) if table is None else table
    if MY_CLIENTS:
        t = t.filter(pc.is_in(t.column("client_hash_id"),
                              value_set=pa.array(sorted(MY_CLIENTS))))
    t = t.filter(pc.fill_null(t.column("gsc_data_available"), False))
    t = t.filter(pc.greater(t.column("gsc_impressions"), 0))
    g = t.group_by(KEYS).aggregate([("report_date", "count"),
                                    ("gsc_impressions", "sum"),
                                    ("gsc_clicks", "sum"),
                                    ("gsc_sum_position", "sum")]).to_pandas()
    g = g.rename(columns={"report_date_count": "active_days",
                          "gsc_impressions_sum": "impr",
                          "gsc_clicks_sum": "clicks",
                          "gsc_sum_position_sum": "sum_pos"})
    g["month"] = m
    _AGG[m] = g
    return g


def count_true(mask_col):
    return int(pc.sum(pc.cast(mask_col, pa.int64())).as_py() or 0)


print("\nready. nothing has been read from the warehouse yet.")

SystemExit: could not find the repo root from /content

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row of `fact_content_daily_performance` is one page, on one day, for one client. That's the
grain, and the first query in Section 3 checks it instead of me assuming it.

The row I actually model on is coarser: **one (client, content) pair at one decision point,
2026-03-01.** Everything up to that date is a feature. The month after it is the label.

| | window | days |
|---|---|---:|
| features | 2026-01-01 → 2026-02-28 | 59 |
| decision point | 2026-03-01 | — |
| label | 2026-03-01 → 2026-03-31 | 31 |

Zero shared days, which is the entire point. ML-03's label came out of the same 90 days as its
features and I flagged that as unresolved. This is my attempt to actually resolve it: the
warehouse lets me put the label after the features instead of beside them.

Two things I didn't want to fudge. 59 isn't 60 and 31 isn't 30, so I turn both windows into a
per-30-day rate before comparing them — dividing by a tidy 2.0 would quietly bake a couple of
percent straight into the label. And the panel is unbalanced, so one global calendar window would
read "tracking hadn't started yet" as "no traffic". I keep only clients whose `gsc_data_start` is
on or before 2026-01-01, and the cell below shows what that costs me.

This contract reads exactly three tables: **`fact_content_daily_performance`** for the daily
impressions, clicks and position behind both windows, **`dim_clients`** for who is in the panel and
when their tracking starts, and **`dim_content`** for page creation dates. Nothing else is opened.

Developing on **month=2026-03** plus the two months behind it. I'm not touching `_sample`: I
checked its dates rather than trusting the note, and it runs 2026-06-01 to 2026-06-30, the last
month in the snapshot. April, May and June stay unread for now.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Which clients can honestly fill a 2026-01-01 -> 2026-02-28 feature window?
t0 = time.time()
clients = pq.read_table(fs.open(WAREHOUSE + "/dim_clients.parquet")).to_pandas()
gsc_start = pd.to_datetime(clients["gsc_data_start"].astype("string"))
ga4_start = pd.to_datetime(clients["ga4_data_start"].astype("string"))

print("dim_clients: %d rows, %d distinct clients" % (len(clients), clients["client_hash_id"].nunique()))
print("access_profile:")
print(clients["access_profile"].value_counts().to_string())
print()
print("gsc_data_start  %s .. %s" % (gsc_start.min().date(), gsc_start.max().date()))
print("ga4_data_start  %s .. %s" % (ga4_start.min().date(), ga4_start.max().date()))
print()
print("the access flags are three-valued, which is why IS TRUE matters later:")
for c in ("is_active", "has_gsc_access", "has_ga4_access"):
    print("  %-15s TRUE=%-4d FALSE=%-4d NULL=%d"
          % (c, int((clients[c] == True).sum()), int((clients[c] == False).sum()),  # noqa: E712
             int(clients[c].isna().sum())))

covers_gsc = gsc_start <= FEATURE_START
covers_ga4 = ga4_start <= FEATURE_START
# IS NOT FALSE, not == TRUE: a NULL flag is unknown, not absent, and these clients do
# have rows. The row-level gsc_data_available IS TRUE filter does the real work later.
keep = covers_gsc & (clients["has_gsc_access"] != False)  # noqa: E712
print()
print("clients whose GSC history covers the feature window : %d of %d" % (int(covers_gsc.sum()), len(clients)))
print("  of those, has_gsc_access explicitly FALSE          : %d"
      % int((covers_gsc & (clients["has_gsc_access"] == False)).sum()))  # noqa: E712
print("  kept (IS NOT FALSE)                                : %d" % int(keep.sum()))
print("     ...of which the flag is NULL, not TRUE          : %d"
      % int((keep & clients["has_gsc_access"].isna()).sum()))
print()
print("same cut for GA4 (engagement)                        : %d of %d"
      % (int(covers_ga4.sum()), len(clients)))
MY_CLIENTS = set(clients.loc[keep, "client_hash_id"])
print()
print("=> I build on %d clients. GA4 is a different story, which is Section 4." % len(MY_CLIENTS))
print("(dim_clients read in %.1fs)" % (time.time() - t0))

dim_clients: 104 rows, 104 distinct clients
access_profile:
access_profile
gsc_and_ga4                             53
no_search_or_analytics_access           26
gsc_only                                14
source_only_missing_client_dimension    10
ga4_only                                 1

gsc_data_start  2025-01-27 .. 2026-06-02
ga4_data_start  2025-10-29 .. 2026-06-01

the access flags are three-valued, which is why IS TRUE matters later:
  is_active       TRUE=74   FALSE=20   NULL=10
  has_gsc_access  TRUE=67   FALSE=27   NULL=10
  has_ga4_access  TRUE=54   FALSE=40   NULL=10

clients whose GSC history covers the feature window : 40 of 104
  of those, has_gsc_access explicitly FALSE          : 3
  kept (IS NOT FALSE)                                : 37
     ...of which the flag is NULL, not TRUE          : 4

same cut for GA4 (engagement)                        : 8 of 104

=> I build on 37 clients. GA4 is a different story, which is Section 4.
(dim_clients read in 1.9s)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

The excluded list is the part I'd most like a second opinion on, so every line gets a reason.

**Features — five, and that's the cap.** All of them knowable on 2026-03-01:

1. `log_impr_prior` — log impressions over the 59-day feature window
2. `ctr_prior_pct` — clicks ÷ impressions over the same window, as a percentage
3. `wtd_pos_prior` — impression-weighted mean position over the window
4. `active_days_prior` — how many of the 59 days the page actually showed up
5. `content_age_days` — age at the decision point, from `dim_content`

Section 3.5 has the "knowable because" line for each, and the timing behind it.

**Label / proxy — never a feature.** `impr_label` (March impressions), `prior_rate_30d`,
`label_rate_30d`, `drop_pct`, `lost_impr_30d`, and the binary `is_material_decline`. That last one
is ML-03's rule moved onto real windows: prior-30d rate ≥ 500 and `drop_pct` ≤ −20, inside a pool
of pages with ≥ 200 prior-window impressions.

**Context — for grouping, joining, splitting. Never learned from.** `client_hash_id` (my CV
group), `content_hash_id` (the join and the display key), `content_type`, `word_count`, `month`.
The IDs are salted hashes; the codes mean nothing and would only invite memorising clients.

**Excluded, and why:**

| excluded | why |
|---|---|
| every March metric, incl. `gsc_impressions` from the label month | it *is* the label window; `drop_pct` is computed from it |
| `gsc_data_available`, `ga4_data_available` | filtering scaffolding, not signal |
| all `ga4_*` | only 8 of 104 clients had GA4 live over my feature window — Section 4 |
| `fact_content_query_90d` entirely | its window is 2026-04-02 → 2026-06-30, entirely *after* my decision point |
| `content_updated_date` | measured, not assumed: 78% of my rows carry an update dated after 2026-03-01, so "days since update" would be reading the future |
| `fact_content_daily_performance_sample` | it is 2026-06, the sealed final month |
| product decision fields (`health_score`, `priority_score`, `action_type`) | not shipped, and predicting them would be circular |

I want to be honest about that `content_updated_date` one. I went in expecting to use
`days_since_content_update` — freshness is an obvious Lane 2 signal. I only dropped it because
the cell in Section 3.5 counted the contamination instead of trusting the column name.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Every field I named above has to actually exist, and has to be the column I think it is.
FEATURES = ["log_impr_prior", "ctr_prior_pct", "wtd_pos_prior", "active_days_prior", "content_age_days"]
LABEL_SIDE = ["impr_label", "prior_rate_30d", "label_rate_30d", "drop_pct", "lost_impr_30d",
              "is_material_decline"]
CONTEXT = ["client_hash_id", "content_hash_id", "content_type", "word_count", "month"]
RAW_FEATURE_SOURCES = {
    "log_impr_prior":     "gsc_impressions summed over month=2026-01, month=2026-02",
    "ctr_prior_pct":      "gsc_clicks / gsc_impressions, same months",
    "wtd_pos_prior":      "gsc_sum_position / gsc_impressions, same months",
    "active_days_prior":  "count of days with gsc_impressions > 0, same months",
    "content_age_days":   "dim_content.content_created_date, measured at the decision point",
}
EXCLUDED = {
    "March metrics":              "the label window itself",
    "gsc_data_available":         "filter, not signal",
    "ga4_*":                      "8 of 104 clients had GA4 live over the feature window",
    "fact_content_query_90d":     "window is 2026-04-02 -> 2026-06-30, all after the decision point",
    "content_updated_date":       "78% of rows carry a post-decision-point update date",
    "_sample table":              "it is 2026-06, the sealed final month",
    "health_score etc.":          "not shipped; predicting a product flag is circular",
}

daily_cols = schema_of(month_file("2026-03"))
content_cols = schema_of(WAREHOUSE + "/dim_content.parquet")

print("fact_content_daily_performance has %d columns" % len(daily_cols))
missing = [c for c in list(RAW_FEATURE_SOURCES) + LABEL_SIDE + CONTEXT
           if c not in daily_cols and c not in content_cols
           and c not in ("log_impr_prior", "ctr_prior_pct", "wtd_pos_prior",
                         "active_days_prior", "content_age_days", "is_material_decline",
                         "prior_rate_30d", "label_rate_30d", "drop_pct", "lost_impr_30d")]
print("names I expected to find upstream but did NOT:", missing or "none - all accounted for")
print()
print("%-22s %s" % ("FEATURE (5)", "source column(s)"))
for f_, s_ in RAW_FEATURE_SOURCES.items():
    print("  %-20s %s" % (f_, s_))
print()
print("  LABEL / PROXY (%d): %s" % (len(LABEL_SIDE), ", ".join(LABEL_SIDE)))
print("  CONTEXT (%d):       %s" % (len(CONTEXT), ", ".join(CONTEXT)))
print()
print("EXCLUDED, with the reason:")
for k, v in EXCLUDED.items():
    print("  %-26s %s" % (k, v))
print()
print("gsc_data_available is present in the daily fact: %s" % ("gsc_data_available" in daily_cols))
print("ga4 availability flags present: %s"
      % [c for c in daily_cols if "available" in c])

# I claimed _sample is the sealed 2026-06 month, so check that instead of trusting the note.
# Read straight from the Parquet footer, so this costs no data I/O at all: report_date
# statistics and nothing else. No metric, no client, no row body is ever touched.
SAMPLE_PATH = WAREHOUSE + "/fact_content_daily_performance_sample.parquet"
_sf = pq.ParquetFile(fs.open(SAMPLE_PATH))
_smd = _sf.metadata
_si = _smd.schema.names.index("report_date")
_s_min = min(_smd.row_group(g).column(_si).statistics.min for g in range(_smd.num_row_groups))
_s_max = max(_smd.row_group(g).column(_si).statistics.max for g in range(_smd.num_row_groups))
print()
print("_sample date check - from the Parquet footer, no rows read:")
print("  rows in file      : %s over %d row groups" % (f"{_smd.num_rows:,}", _smd.num_row_groups))
print("  report_date range : %s -> %s" % (_s_min, _s_max))
print("  that is           : %s" % ("2026-06, the sealed final month - correctly excluded"
                                    if (str(_s_min), str(_s_max)) == ("2026-06-01", "2026-06-30")
                                    else "NOT 2026-06, so this exclusion needs re-reading"))
print("  overlaps my frame : %s  (my frame is %s -> %s)"
      % (str(_s_max) >= FEATURE_START and str(_s_min) <= LABEL_END, FEATURE_START, LABEL_END))
print("  columns read      : 0 (statistics only)")
del _sf, _smd

fact_content_daily_performance has 30 columns
names I expected to find upstream but did NOT: ['impr_label', 'month']

FEATURE (5)            source column(s)
  log_impr_prior       gsc_impressions summed over month=2026-01, month=2026-02
  ctr_prior_pct        gsc_clicks / gsc_impressions, same months
  wtd_pos_prior        gsc_sum_position / gsc_impressions, same months
  active_days_prior    count of days with gsc_impressions > 0, same months
  content_age_days     dim_content.content_created_date, measured at the decision point

  LABEL / PROXY (6): impr_label, prior_rate_30d, label_rate_30d, drop_pct, lost_impr_30d, is_material_decline
  CONTEXT (5):       client_hash_id, content_hash_id, content_type, word_count, month

EXCLUDED, with the reason:
  March metrics              the label window itself
  gsc_data_available         filter, not signal
  ga4_*                      8 of 104 clients had GA4 live over the feature window
  fact_content_query_90d     window is 2026-04-02 -> 2


_sample date check - from the Parquet footer, no rows read:
  rows in file      : 11,694,072 over 98 row groups
  report_date range : 2026-06-01 -> 2026-06-30
  that is           : 2026-06, the sealed final month - correctly excluded
  overlaps my frame : False  (my frame is 2026-01-01 -> 2026-03-31)
  columns read      : 0 (statistics only)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries on the mid-panel month, `month=2026-03`, before I build anything:

1. **Grain.** Does one row really mean one page on one day for one client? A
   `GROUP BY report_date, client, content HAVING COUNT(*) > 1` that comes back empty.
2. **Row count and date span** of the slice I actually intend to use — my 37 clients, GSC-available
   days only.
3. **Availability, filtered `IS TRUE`** — and what the other two ways of writing that filter would
   have swept in.

I'm using the flag rather than the metric columns, because the dictionary says rows before a
client's `ga4_data_start` are *zero-filled*, and NULL-flagged rows are *neither* zero-filled nor
FALSE. Query 3 measures the difference rather than quoting the warning at you — though I should say
up front that `month=2026-03` is a month where the NULL count comes back zero, so to show the
three-valued behaviour actually biting I count NULLs in the two feature-window months as well.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Three verification queries on month=2026-03. Nothing is built yet.
print("=" * 78)
print("QUERY 1 - is the grain really (report_date, client, content)?")
print("=" * 78)
t0 = time.time()
t = read_month(DEV_MONTH)
print("month=%s raw rows: %s" % (DEV_MONTH, f"{t.num_rows:,}"))

g = t.group_by(["report_date"] + KEYS).aggregate([("gsc_impressions", "count")])
dupe_count = g.column("gsc_impressions_count")
n_grain_breaks = count_true(pc.greater(dupe_count, 1))
print("distinct (report_date, client, content) combinations: %s" % f"{len(g):,}")
print("combinations appearing more than once                 : %d" % n_grain_breaks)
print("grain holds:", n_grain_breaks == 0)
if n_grain_breaks:
    print(g.filter(pc.greater(dupe_count, 1)).slice(0, 5).to_pandas().to_string())
del g, dupe_count
print("(%.1fs)" % (time.time() - t0))

print()
print("=" * 78)
print("QUERY 2 - row count and date span of the slice I mean to use")
print("=" * 78)
avail = pc.fill_null(t.column("gsc_data_available"), False)      # IS TRUE
mine = pc.is_in(t.column("client_hash_id"), value_set=pa.array(sorted(MY_CLIENTS)))
slice_mask = pc.and_(avail, mine)
sl = t.filter(slice_mask)
print("slice = month=%s, my %d clients, gsc_data_available IS TRUE" % (DEV_MONTH, len(MY_CLIENTS)))
print("  rows          : %s  (%.1f%% of the partition)"
      % (f"{sl.num_rows:,}", 100 * sl.num_rows / t.num_rows))
print("  report_date   : %s .. %s" % (sl.column("report_date")[0].as_py(), sl.column("report_date")[-1].as_py()))
print("  distinct pages: %d page-clients" % len(set(zip(sl.column("client_hash_id").to_pylist(),
                                                        sl.column("content_hash_id").to_pylist()))))
print("  distinct dates: %d" % len(set(sl.column("report_date").to_pylist())))
print("  this is one month of daily rows, not one row per page - Section 3.5 collapses it.")

print()
print("=" * 78)
print("QUERY 3 - availability, filtered IS TRUE, and what the alternatives cost")
print("=" * 78)
flag = t.column("gsc_data_available")
n_total = t.num_rows
n_true = count_true(avail)
n_null = count_true(pc.is_null(flag))
n_false = n_total - n_true - n_null
print("gsc_data_available across month=%s, all %s rows:" % (DEV_MONTH, f"{n_total:,}"))
print("  IS TRUE     : %s  (%.1f%%)   <- the filter I use" % (f"{n_true:,}", 100 * n_true / n_total))
print("  IS FALSE    : %s  (%.1f%%)" % (f"{n_false:,}", 100 * n_false / n_total))
print("  IS NULL     : %s  (%.1f%%)   <- the flag is three-valued, so '= FALSE' is not 'not TRUE'"
      % (f"{n_null:,}", 100 * n_null / n_total))
print("                ...but it happens to be 0 in %s, so this month cannot show you the" % DEV_MONTH)
print("                difference. Counting NULLs in the two feature-window months instead:")

# The same count on the months where the third state actually shows up. Cheap: two columns.
NULLS_BY_MONTH = {}
for _m in FEATURE_MONTHS:
    _t = read_month(_m, cols=["gsc_data_available", "gsc_impressions"])
    _fl = _t.column("gsc_data_available")
    _n_null = count_true(pc.is_null(_fl))
    _n_both = count_true(pc.and_(pc.is_null(_fl), pc.is_null(_t.column("gsc_impressions"))))
    NULLS_BY_MONTH[_m] = (_n_null, _n_both)
    print("                  month=%s  IS NULL = %s  (%.1f%% of %s rows) | %s of those also have NULL impressions"
          % (_m, f"{_n_null:,}", 100 * _n_null / _t.num_rows, f"{_t.num_rows:,}", f"{_n_both:,}"))
    del _t, _fl
print("                so the three-valued behaviour is real and it bites in the feature window;")
print("                month=%s just happens to be a month where it is absent." % DEV_MONTH)

# What the two lazy filters would have done, measured on the metrics themselves.
n_null_metric_null = count_true(pc.and_(
    pc.is_null(flag), pc.is_null(t.column("gsc_impressions"))))
is_absent = pc.invert(pc.fill_null(flag, False))
false_block = t.filter(is_absent)
fb_imp = false_block.column("gsc_impressions")
n_false_zero = count_true(pc.equal(fb_imp, 0))
n_false_null = count_true(pc.is_null(fb_imp))
print()
print("what sits behind each flag (this is the reason IS TRUE is not optional):")
print("  IS FALSE rows  : gsc_impressions == 0 in %s of %s, NULL in %s  -> zero-filled, not 'absent'"
      % (f"{n_false_zero:,}", f"{len(fb_imp):,}", f"{n_false_null:,}"))
print("  IS NULL rows   : %s in %s, of which %s also carry NULL impressions"
      % (f"{n_null:,}", DEV_MONTH, f"{n_null_metric_null:,}"))
print()
print("  a filter of '= TRUE' happens to agree with IS TRUE here, because NULL never equals")
print("  TRUE. The dangerous one is 'NOT gsc_data_available' or '= FALSE': both quietly treat")
print("  NULL-flagged rows as usable, and those carry NULL metrics, which turn into NULL sums")
print("  that either vanish or get silently filled later.")
print()
_worst = max(NULLS_BY_MONTH, key=lambda m: NULLS_BY_MONTH[m][0])
print("  In %s that set is %s rows, so the mistake would have cost me nothing *this* month."
      % (DEV_MONTH, f"{n_null:,}"))
print("  In %s it is not: %s rows flagged NULL, %s of them with NULL impressions. An '= FALSE'"
      % (_worst, f"{NULLS_BY_MONTH[_worst][0]:,}", f"{NULLS_BY_MONTH[_worst][1]:,}"))
print("  filter would have pulled every one of those in as if they were real zeros - and those")
print("  are exactly the %s days my feature window is built from." % f"{PRIOR_DAYS}")
print("  So the filter is IS TRUE, and I do not demonstrate it on %s alone." % DEV_MONTH)
del false_block, fb_imp

# Cross-check my aggregate position against the shipped daily average, while March is in memory.
chk = t.filter(avail).select(["gsc_impressions", "gsc_sum_position", "gsc_avg_position"]).to_pandas()
chk = chk[chk["gsc_impressions"] > 0]
mine_pos = chk["gsc_sum_position"] / chk["gsc_impressions"]
print()
print("sanity: sum_position / impressions vs the shipped gsc_avg_position (available rows)")
print("  rows compared : %s" % f"{len(chk):,}")
print("  max abs diff  : %.6f" % float((mine_pos - chk["gsc_avg_position"]).abs().max()))
print("  mean position : %.3f" % float(chk["gsc_avg_position"].mean()))
del chk

# Memoise the March aggregate from the table already in memory - no second read.
agg_month(DEV_MONTH, table=t)
del t, sl
print()
print("March aggregated to %s page-clients and cached for Section 3.5." % f"{len(_AGG[DEV_MONTH]):,}")
print("total so far: %.1fs" % (time.time() - t0))

QUERY 1 - is the grain really (report_date, client, content)?


month=2026-03 raw rows: 9,841,378


distinct (report_date, client, content) combinations: 9,841,378
combinations appearing more than once                 : 0
grain holds: True
(141.0s)

QUERY 2 - row count and date span of the slice I mean to use


slice = month=2026-03, my 37 clients, gsc_data_available IS TRUE
  rows          : 3,045,934  (31.0% of the partition)
  report_date   : 2026-03-01 .. 2026-03-31


  distinct pages: 145190 page-clients
  distinct dates: 31
  this is one month of daily rows, not one row per page - Section 3.5 collapses it.

QUERY 3 - availability, filtered IS TRUE, and what the alternatives cost


...

                  month=2026-01  IS NULL = 5,750  (0.1% of 7,890,817 rows) | 5,750 of those also have NULL impressions


                  month=2026-02  IS NULL = 92,256  (1.3% of 7,355,108 rows) | 92,256 of those also have NULL impressions
                so the three-valued behaviour is real and it bites in the feature window;
                month=2026-03 just happens to be a month where it is absent.



what sits behind each flag (this is the reason IS TRUE is not optional):
  IS FALSE rows  : gsc_impressions == 0 in 6,230,317 of 6,230,317, NULL in 0  -> zero-filled, not 'absent'
  IS NULL rows   : 0 in 2026-03, of which 0 also carry NULL impressions

  a filter of '= TRUE' happens to agree with IS TRUE here, because NULL never equals
  TRUE. The dangerous one is 'NOT gsc_data_available' or '= FALSE': both quietly treat
  NULL-flagged rows as usable, and those carry NULL metrics, which turn into NULL sums
  that either vanish or get silently filled later.

  In 2026-03 that set is 0 rows, so the mistake would have cost me nothing *this* month.
  In 2026-02 it is not: 92,256 rows flagged NULL, 92,256 of them with NULL impressions. An '= FALSE'
  filter would have pulled every one of those in as if they were real zeros - and those
  are exactly the 59 days my feature window is built from.
  So the filter is IS TRUE, and I do not demonstrate it on 2026-03 alone.



sanity: sum_position / impressions vs the shipped gsc_avg_position (available rows)
  rows compared : 3,611,061
  max abs diff  : 0.000000
  mean position : 15.827



March aggregated to 145,190 page-clients and cached for Section 3.5.
total so far: 327.6s


### 3.5 The five features, and why each one is knowable

Now I collapse the daily rows into one row per (client, content) and build the frame. The rule I
applied to every feature: *knowable at 2026-03-01 because...*

| feature | knowable at the decision moment because |
|---|---|
| `log_impr_prior` | it sums `gsc_impressions` for 2026-01-01 → 02-28 only; every row is dated before the decision point |
| `ctr_prior_pct` | clicks and impressions from those same 59 days, same rows |
| `wtd_pos_prior` | `gsc_sum_position / gsc_impressions` over those 59 days — impression-weighted, and it matches the shipped daily `gsc_avg_position` to 6 decimals (Query 3) |
| `active_days_prior` | it counts days in that window with impressions > 0, so it can only look backwards |
| `content_age_days` | `dim_content.content_created_date` measured *at* 2026-03-01. Pages can't be created after they existed, and I confirmed 0 rows have a creation date past the decision point |

Two features I wanted and dropped:

- **`days_since_content_update`.** The obvious freshness signal, and it's a trap. `dim_content` is
  a current snapshot, not an as-of-decision-point table: 93,162 of my 119,123 rows (78%) carry a
  `content_updated_date` *after* 2026-03-01. Using it would have been reading the future and
  calling it freshness.
- **Anything from `fact_content_query_90d`.** Its window is 2026-04-02 → 2026-06-30. It looks
  useful — it even has `*_prev30` columns the dictionary blesses for exactly this kind of label —
  but every window it carries sits downstream of my decision point, so nothing in it is a feature
  here. The dictionary's own advice (`*_prev30` is safe) only holds for a label on the *final*
  month, which is not the label I'm building.

`active_days_prior` deserves a caveat: its median in the pool is 59, i.e. most pages
showed up every single day, so it does nothing for the typical page and only separates the
intermittent tail. I kept it because the tail is real, not because it carried much weight.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Build the one-row-per-(client, content) frame and the five features.
t0 = time.time()
for m in FEATURE_MONTHS:
    if m not in _AGG:
        print("aggregating %s ..." % m)
        agg_month(m)

prior = pd.concat([_AGG[m] for m in FEATURE_MONTHS])
label = _AGG[LABEL_MONTHS[0]][["client_hash_id", "content_hash_id", "impr", "active_days"]]
label = label.rename(columns={"impr": "impr_label", "active_days": "active_days_label"})
prior = prior.groupby(KEYS, as_index=False).agg(
    impr_prior=("impr", "sum"), clicks_prior=("clicks", "sum"),
    sum_pos_prior=("sum_pos", "sum"), active_days_prior=("active_days", "sum"))

n_prior_only = len(prior) - len(prior.merge(label, on=KEYS, how="inner"))
f = prior.merge(label, on=KEYS, how="inner")
print("page-clients in the feature window : %s" % f"{len(prior):,}")
print("  also present in the label window: %s" % f"{len(f):,}")
print("  dropped (no March rows to label) : %s  <- pages that stopped reporting" % f"{n_prior_only:,}")

# Per-30-day rates. 59 and 31 are not 60 and 30, so I normalise both sides.
f["prior_rate_30d"] = f["impr_prior"] / PRIOR_DAYS * 30
f["label_rate_30d"] = f["impr_label"] / LABEL_DAYS * 30
f["drop_pct"] = np.where(f["prior_rate_30d"] > 0,
                         (f["label_rate_30d"] - f["prior_rate_30d"]) / f["prior_rate_30d"].replace(0, np.nan) * 100,
                         np.nan)
f["lost_impr_30d"] = f["prior_rate_30d"] - f["label_rate_30d"]

# ---- the five features ----------------------------------------------------
f["log_impr_prior"] = np.log1p(f["impr_prior"])
f["ctr_prior_pct"] = np.where(f["impr_prior"] > 0, f["clicks_prior"] / f["impr_prior"] * 100, np.nan)
f["wtd_pos_prior"] = np.where(f["impr_prior"] > 0, f["sum_pos_prior"] / f["impr_prior"], np.nan)

content = pq.read_table(fs.open(WAREHOUSE + "/dim_content.parquet"),
                        columns=["content_hash_id", "content_created_date", "content_updated_date"]).to_pandas()
content["content_created_date"] = pd.to_datetime(content["content_created_date"].astype("string"))
content["content_updated_date"] = pd.to_datetime(content["content_updated_date"].astype("string"))
f = f.merge(content, on="content_hash_id", how="left")
f["content_age_days"] = (pd.Timestamp(DECISION_POINT) - f["content_created_date"]).dt.days

# ---- the label: ML-03's rule, on real forward windows ---------------------
POOL_FLOOR, VOLUME_FLOOR, DROP_FLOOR = 200, 500, -20
f["in_pool"] = f["impr_prior"] >= POOL_FLOOR
f["is_material_decline"] = (f["in_pool"] & (f["prior_rate_30d"] >= VOLUME_FLOOR)
                            & (f["drop_pct"] <= DROP_FLOOR)).astype(int)

print()
print("=== frame ===")
print("rows %s | clients %d | drop_pct NaN %d"
      % (f"{len(f):,}", f["client_hash_id"].nunique(), int(f["drop_pct"].isna().sum())))
P = f[f["in_pool"]]
print("pool (>= %d prior impressions): %s rows (%.1f%%)"
      % (POOL_FLOOR, f"{len(P):,}", 100 * len(P) / len(f)))
print("material decline: %s = %.4f of the pool  [rate >= %d/mo AND drop_pct <= %d%%]"
      % (f"{int(P['is_material_decline'].sum()):,}", P["is_material_decline"].mean(), VOLUME_FLOOR, DROP_FLOOR))
print("drop_pct in pool: median %.1f%% | p10 %.1f%% | p90 %.1f%%"
      % (P["drop_pct"].median(), P["drop_pct"].quantile(.1), P["drop_pct"].quantile(.9)))

print()
print("=== the five features, on the pool ===")
print("  %-18s %7s %10s %10s %10s" % ("feature", "n", "median", "p90", "max"))
for c in FEATURES:
    print("  %-18s %7d %10.2f %10.2f %10.2f"
          % (c, P[c].notna().sum(), P[c].median(), P[c].quantile(.9), P[c].max()))

print()
print("=== availability timing, measured ===")
print("  content_created_date after the decision point : %d  (age is safe by construction)"
      % int((f["content_created_date"] > DECISION_POINT).sum()))
print("  content_updated_date after the decision point : %s of %s (%.0f%%)  <- freshness is NOT safe"
      % (f"{int((f['content_updated_date'] > DECISION_POINT).sum()):,}", f"{len(f):,}",
         100 * (f["content_updated_date"] > DECISION_POINT).mean()))
print("  active_days_prior == all %d days for %.0f%% of the pool (saturated; only the tail moves)"
      % (PRIOR_DAYS, 100 * (P["active_days_prior"] == PRIOR_DAYS).mean()))
print()
print("built in %.1fs" % (time.time() - t0))

aggregating 2026-01 ...


aggregating 2026-02 ...


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month%3D2026-02/data_0.parquet


Retrying in 1s [Retry 1/5].


page-clients in the feature window : 138,256
  also present in the label window: 119,123
  dropped (no March rows to label) : 19,133  <- pages that stopped reporting


...

'[Errno 11001] getaddrinfo failed' thrown while requesting GET https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/dim_content.parquet


Retrying in 4s [Retry 3/5].


'[Errno 11001] getaddrinfo failed' thrown while requesting GET https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/dim_content.parquet


Retrying in 8s [Retry 4/5].



=== frame ===
rows 119,123 | clients 29 | drop_pct NaN 0
pool (>= 200 prior impressions): 67,279 rows (56.5%)
material decline: 7,445 = 0.1107 of the pool  [rate >= 500/mo AND drop_pct <= -20%]
drop_pct in pool: median 26.3% | p10 -39.0% | p90 217.2%

=== the five features, on the pool ===
  feature                  n     median        p90        max
  log_impr_prior       67279       7.27       9.24      12.81
  ctr_prior_pct        67279       0.17       0.63      21.95
  wtd_pos_prior        67279       6.97      23.38      94.56
  active_days_prior    67279      59.00      59.00      59.00
  content_age_days     67279     200.00     366.00     464.00

=== availability timing, measured ===
  content_created_date after the decision point : 0  (age is safe by construction)
  content_updated_date after the decision point : 93,162 of 119,123 (78%)  <- freshness is NOT safe
  active_days_prior == all 59 days for 64% of the pool (saturated; only the tail moves)

built in 354.5s


### 3.6 The trap, on purpose

ML-03 proved leakage by describing it. The warehouse lets me do it for real, so let me.

I'm going to add **one** column to the honest five: `log_impr_label`, the log of March
impressions. It looks harmless — "recent impressions" is the kind of column that ends up in real
feature lists — but March is the label window, and `drop_pct` is a function of it. A model handed
this can read the answer off the input.

I fit both, client-grouped, five folds, and print both. Then I delete the column and keep the
honest number.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Deliberate leak: one label-derived column, both versions actually run.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

LEAK_COL = "log_impr_label"
f[LEAK_COL] = np.log1p(f["impr_label"])          # March impressions == the label window

y = f.loc[f["in_pool"], "is_material_decline"].to_numpy()
groups = f.loc[f["in_pool"], "client_hash_id"].to_numpy()
X_honest = f.loc[f["in_pool"], FEATURES].to_numpy(dtype=float)
X_leaked = f.loc[f["in_pool"], FEATURES + [LEAK_COL]].to_numpy(dtype=float)
SEED = 0


def oof_scores(X):
    """Out-of-fold probabilities, grouped by client so no client spans train and test."""
    out = {}
    for name, mdl in {
        "logistic regression": Pipeline([("scale", StandardScaler()),
                                         ("clf", LogisticRegression(max_iter=2000))]),
        "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                               random_state=SEED, n_jobs=-1),
    }.items():
        s = np.zeros(len(y))
        for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
            mdl.fit(X[tr], y[tr])
            s[te] = mdl.predict_proba(X[te])[:, 1]
        out[name] = s
    return out


def p_at_k(yy, scores, k=50):
    return float(np.asarray(yy)[np.argsort(-np.asarray(scores), kind="stable")[:k]].mean())


BASE = float(y.mean())
print("pool %s rows | %s positives | base rate %.4f | %d clients in %d folds"
      % (f"{len(y):,}", f"{int(y.sum()):,}", BASE, len(set(groups)), 5))
print("P@50 for a random queue is just the base rate, so %.3f is the floor." % BASE)
print()
print("%-34s %7s %7s" % ("", "AUC", "P@50"))
print("HONEST - the five features, nothing from the label window")
honest = oof_scores(X_honest)
for n, s in honest.items():
    print("  %-32s %7.3f %7.3f" % (n, roc_auc_score(y, s), p_at_k(y, s)))
print()
print("LEAKED - same model, plus %s" % LEAK_COL)
leaked = oof_scores(X_leaked)
for n, s in leaked.items():
    print("  %-32s %7.3f %7.3f" % (n, roc_auc_score(y, s), p_at_k(y, s)))

print()
print("what just happened:")
print("  adding March impressions took the forest to P@50 = %.3f. A reviewer would call that a"
      % p_at_k(y, leaked["random forest"]))
print("  breakthrough. It is arithmetic: drop_pct is monotone in the March impression count, so")
print("  the model is not predicting decline, it is reading the decline off the input.")
print()
print("=> deleting the column and keeping the honest number")
del f[LEAK_COL]
kept = f[f["in_pool"]][FEATURES].to_numpy(dtype=float)
print("   feature list is now: %s" % ", ".join(FEATURES))
print("   no column in it is derived from %s -> %s"
      % (LABEL_START, LABEL_END))
print("   the number I stand behind: P@50 %.3f, AUC %.3f (random forest, client-grouped)"
      % (p_at_k(y, honest["random forest"]), roc_auc_score(y, honest["random forest"])))
print("   that is %.1fx the base rate of %.3f, from five features that predate the label."
      % (p_at_k(y, honest["random forest"]) / BASE, BASE))
print()
print("Still to do before I'd trust that number: a time-aware split, a per-client breakdown, and")
print("a look at the pages sitting just under the top of the queue. Section 3.6 is evidence that")
print("the frame is clean, not a claim that the model is finished.")

pool 67,279 rows | 7,445 positives | base rate 0.1107 | 24 clients in 5 folds
P@50 for a random queue is just the base rate, so 0.111 is the floor.

                                       AUC    P@50
HONEST - the five features, nothing from the label window


  logistic regression                0.754   0.460
  random forest                      0.793   0.560

LEAKED - same model, plus log_impr_label


  logistic regression                0.949   0.580
  random forest                      0.997   1.000

what just happened:
  adding March impressions took the forest to P@50 = 1.000. A reviewer would call that a
  breakthrough. It is arithmetic: drop_pct is monotone in the March impression count, so
  the model is not predicting decline, it is reading the decline off the input.

=> deleting the column and keeping the honest number
   feature list is now: log_impr_prior, ctr_prior_pct, wtd_pos_prior, active_days_prior, content_age_days
   no column in it is derived from 2026-03-01 -> 2026-03-31
   the number I stand behind: P@50 0.560, AUC 0.793 (random forest, client-grouped)
   that is 5.1x the base rate of 0.111, from five features that predate the label.

Still to do before I'd trust that number: a time-aware split, a per-client breakdown, and
a look at the pages sitting just under the top of the queue. Section 3.6 is evidence that
the frame is clean, not a claim that the model is f

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**The named limitation: engagement is not really available for this window, so my frame is
GSC-only.**

My label is a traffic decline, which is the right label for Lane 2. But the natural way to
prioritise a refresh is to combine "losing traffic" with "people aren't engaging with it" — and
that's exactly what's missing. Of 104 clients, **8** had GA4 history live on 2026-01-01. Build the
same frame on GA4 sessions or engagement rate and I'm working with those 8, which is not enough
clients to say anything I'd want to publish. So `ga4_sessions`, `ga4_engaged_sessions`,
`ga4_pageviews` and the engagement rates are all excluded, and this contract is search-visibility
only.

That is specific to what I queried, not a general complaint about the release — GA4 coverage runs
2025-10-29 onward, so an earlier decision point would have had more of it. It's a consequence of
choosing a mid-panel month, which is the right trade for keeping June sealed.

Smaller ones I'm carrying:

- **19,133 page-clients vanished between the windows** — present in Jan–Feb, no March rows. Those
  are pages that stopped reporting. I drop them because I can't label them, which quietly biases
  the pool toward pages that stayed alive.
- **Only 24 of my 37 clients ever reach the pool.** 29 of the 37 have a page in both windows at
  all, and 5 of those have no page clearing 200 prior impressions — so 13 of my 37 are outside the
  frame I train on.
- **The label is a proxy.** "Lost 20% of impressions while still getting 500 a month" is a policy
  I chose, not a measured definition of decline, and ML-04's sibling checks in ML-07/ML-08 should
  move those two numbers and watch what happens.
- **No causal claim is available.** Nothing here can tell me a refresh would have worked.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Evidence for the named limitation: how much engagement data is actually there?
t0 = time.time()
avail_cols = ["client_hash_id", "gsc_data_available", "ga4_data_available"]
raw = read_month(FEATURE_MONTHS[0], cols=avail_cols)
print("month=%s, rows %s (3 narrow columns only)" % (FEATURE_MONTHS[0], f"{raw.num_rows:,}"))

gsc_flag = pc.fill_null(raw.column("gsc_data_available"), False)
ga4_flag = pc.fill_null(raw.column("ga4_data_available"), False)
n = raw.num_rows
print()
print("  gsc_data_available IS TRUE : %s (%.1f%%)" % (f"{count_true(gsc_flag):,}", 100 * count_true(gsc_flag) / n))
print("  ga4_data_available IS TRUE : %s (%.1f%%)" % (f"{count_true(ga4_flag):,}", 100 * count_true(ga4_flag) / n))
print()
print("  so the search half of this fact table is roughly %.0fx denser than the"
      % (count_true(gsc_flag) / max(count_true(ga4_flag), 1)))
print("  engagement half, in the first month of my feature window.")

both = raw.filter(pc.and_(gsc_flag, ga4_flag))
print("  rows with BOTH available: %s of %s (%.1f%%)"
      % (f"{both.num_rows:,}", f"{n:,}", 100 * both.num_rows / n))
del both, raw

print()
print("=== per-client GA4 coverage ===")
print("clients with GA4 history live at the feature-window start: %d of %d"
      % (int(covers_ga4.sum()), len(clients)))
print("clients I am using (GSC-covered):                        %d" % len(MY_CLIENTS))
ga4_ok = set(clients.loc[covers_ga4, "client_hash_id"])
print("...of those, how many also have GA4:                      %d" % len(MY_CLIENTS & ga4_ok))
print()
print("=> an engagement-based version of this frame would be built on %d clients."
      % len(MY_CLIENTS & ga4_ok))
print("   Not enough to claim anything from, so engagement stays out. Named limitation, and it")
print("   is a property of this window rather than of the release: GA4 coverage starts")
print("   %s, so an earlier decision point would have had more of it."
      % ga4_start.min().date())
print()
print("=== what the frame actually ended up as ===")
print("rows %s | pool %s | clients in frame %d | clients in pool %d"
      % (f"{len(f):,}", f"{int(f['in_pool'].sum()):,}", f["client_hash_id"].nunique(),
         f.loc[f["in_pool"], "client_hash_id"].nunique()))
print("(%.1fs)" % (time.time() - t0))

month=2026-01, rows 7,890,817 (3 narrow columns only)

  gsc_data_available IS TRUE : 2,397,143 (30.4%)


  ga4_data_available IS TRUE : 115,786 (1.5%)

  so the search half of this fact table is roughly 21x denser than the
  engagement half, in the first month of my feature window.
  rows with BOTH available: 100,259 of 7,890,817 (1.3%)

=== per-client GA4 coverage ===
clients with GA4 history live at the feature-window start: 8 of 104
clients I am using (GSC-covered):                        37
...of those, how many also have GA4:                      8

=> an engagement-based version of this frame would be built on 8 clients.
   Not enough to claim anything from, so engagement stays out. Named limitation, and it
   is a property of this window rather than of the release: GA4 coverage starts
   2025-10-29, so an earlier decision point would have had more of it.

=== what the frame actually ended up as ===
rows 119,123 | pool 67,279 | clients in frame 29 | clients in pool 24
(135.4s)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card.